# ThriveLens Food-R1 — dual-T4 global food fine-tuning

This notebook fine-tunes `zy12123/Food-R1` on the supplied, real-image ThriveLens dataset with both Kaggle GPUs. It uses 4-bit NF4 QLoRA, FP16 compute (appropriate for T4), gradient checkpointing, distributed data parallelism through DeepSpeed ZeRO-2, deterministic train/validation/test splits, fixed epochs, checkpoint/resume, and a full held-out evaluation. **There is no early stopping.**

Why the vision tower is frozen: current QLoRA implementations do not safely combine 4-bit quantization with vision-layer training. Food-R1 already supplies the food-specialized visual encoder; the adapter trains its language/reasoning path on the new worldwide food recognition and Nutrition5k supervision. Full vision tuning requires substantially more VRAM than two 16-GB T4s.

Kaggle setup: attach this ZIP as a Kaggle Dataset, select **GPU T4 x2**, enable Internet for the one-time model/dependency download, and run all cells. Kaggle's accelerator quota is 30 GPU hours per week; an individual session may still have its own runtime limit.


In [ ]:
# Configuration — change only these if you deliberately want a different run.
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

MODEL_ID = "zy12123/Food-R1"
EPOCHS = 3                 # fixed; no early stopping
MICRO_BATCH = 1            # per GPU
GRAD_ACCUM = 16            # global effective batch = 1 x 2 GPUs x 16 = 32
LEARNING_RATE = 1e-4
LORA_RANK = 32
LORA_ALPHA = 64
MAX_SEQUENCE_LENGTH = 768
MIN_IMAGE_PIXELS = 256 * 256
MAX_IMAGE_PIXELS = 512 * 512
MAX_TEST_SAMPLES = 5000      # set to 0 for the entire large held-out split
SEED = 2026
OUTPUT_DIR = Path("/kaggle/working/thrivelens_foodr1_qlora")
WORK = Path("/kaggle/working/thrivelens_setup")
WORK.mkdir(parents=True, exist_ok=True)
print({"model": MODEL_ID, "epochs": EPOCHS, "global_batch": 2 * MICRO_BATCH * GRAD_ACCUM})


In [ ]:
# Hard fail unless Kaggle assigned exactly the requested two CUDA GPUs.
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
assert torch.cuda.device_count() == 2, f"Select GPU T4 x2; found {torch.cuda.device_count()} GPU(s)."
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 2**30:.1f} GiB", "bf16=", torch.cuda.is_bf16_supported())


In [ ]:
# Find the attached package and extract only code/manifests — images stay inside the ZIP.
candidates = list(Path("/kaggle/input").rglob("ThriveLens_OpenFood_FoodR1_Kaggle*.zip"))
if not candidates:
    candidates = list(Path("/kaggle/input").rglob("*.zip"))
assert len(candidates) == 1, f"Expected one ThriveLens ZIP; found: {candidates}"
ARCHIVE = candidates[0]
print("Archive:", ARCHIVE, f"({ARCHIVE.stat().st_size / 2**30:.2f} GiB)")
import hashlib
archive_hash = hashlib.sha256()
with ARCHIVE.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        archive_hash.update(chunk)
archive_sha256 = archive_hash.hexdigest()
print("SHA-256:", archive_sha256)

with zipfile.ZipFile(ARCHIVE) as zf:
    wanted = [n for n in zf.namelist() if n.startswith(("training/", "sft/")) or n in {
        "README.md", "dataset_manifest.json", "SHA256SUMS.txt"
    }]
    for name in wanted:
        zf.extract(name, WORK)
    # Decode ten representative images now so a corrupt package fails before model download.
    image_names = [n for n in zf.namelist() if n.lower().endswith((".jpg", ".png"))]
    from PIL import Image
    import io
    for name in image_names[::max(1, len(image_names)//10)][:10]:
        with Image.open(io.BytesIO(zf.read(name))) as im:
            im.verify()
print(f"Extracted {len(wanted)} small support files; verified 10 distributed image samples.")

os.environ["THRIVELENS_ARCHIVE"] = str(ARCHIVE)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
# Inspect exact source counts and split/task balance.
manifest = json.loads((WORK / "dataset_manifest.json").read_text())
print(json.dumps(manifest["summary"], indent=2))
for split in ("train", "val", "test"):
    rows = json.loads((WORK / "sft" / f"{split}.json").read_text())
    by_task = {}
    for row in rows:
        by_task[row["task"]] = by_task.get(row["task"], 0) + 1
    print(split, len(rows), by_task)


In [ ]:
# Install the versions pinned with the bundled training implementation.
# We deliberately use SDPA, not FlashAttention, for maximum Kaggle T4 compatibility.
%pip install -q --upgrade \
  "transformers==5.3.0" "accelerate==1.10.1" "peft==0.15.2" \
  "bitsandbytes==0.49.2" "deepspeed==0.17.5" "trl==0.25.0" \
  "qwen-vl-utils" "ujson==5.10.0" "tensorboardX==2.6.2.2" "ninja"


In [ ]:
# Activate the bundled, commit-pinned Qwen3-VL trainer and its ZIP streaming patch.
TRAINER_ROOT = WORK / "training" / "qwen-vl-finetune"
assert TRAINER_ROOT.exists(), TRAINER_ROOT
subprocess.run([sys.executable, str(WORK / "training" / "patch_zip_loader.py")], check=True)

# Confirm this is the source revision validated when this notebook was built.
expected_commit = (TRAINER_ROOT / "PINNED_COMMIT.txt").read_text().strip()
print("Qwen-VL trainer commit:", expected_commit)


In [ ]:
# Validate that all referenced ZIP members exist and that no original image crosses splits.
from collections import Counter
with zipfile.ZipFile(ARCHIVE) as zf:
    names = set(zf.namelist())
    owners = {}
    for split in ("train", "val", "test"):
        rows = json.loads((WORK / "sft" / f"{split}.json").read_text())
        for row in rows:
            images = row["image"] if isinstance(row["image"], list) else [row["image"]]
            for uri in images:
                assert uri.startswith("zip://")
                assert uri[6:] in names, uri
            group = row["group_id"]
            if group in owners:
                assert owners[group] == split, f"Leakage: {group} is in {owners[group]} and {split}"
            owners[group] = split
print(f"Validated every image reference and {len(owners):,} leakage-safe source groups.")


## Train

The command below launches two processes, one per T4. The adapter runs exactly `EPOCHS`; validation loss is measured each epoch but never used to stop training. Checkpoints retain optimizer state and the bundled trainer automatically resumes from the newest `checkpoint-*` directory after a Kaggle interruption.


In [ ]:
# Write a transparent launch script so the exact command is saved with the run output.
launch = f'''#!/usr/bin/env bash
set -euo pipefail
export PYTHONPATH="{TRAINER_ROOT}/src:${{PYTHONPATH:-}}"
export THRIVELENS_ARCHIVE="{ARCHIVE}"
export CUDA_VISIBLE_DEVICES=0,1
export TOKENIZERS_PARALLELISM=false
export WANDB_DISABLED=true
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd "{TRAINER_ROOT}"
deepspeed --num_gpus 2 src/train/train_sft.py \
  --deepspeed scripts/zero2.json \
  --model_id "{MODEL_ID}" \
  --data_path "{WORK / 'sft/train.json'}" \
  --eval_path "{WORK / 'sft/val.json'}" \
  --image_folder /unused/zip_streaming \
  --remove_unused_columns False \
  --bits 4 --double_quant True --quant_type nf4 \
  --lora_enable True --vision_lora False --use_dora False \
  --lora_namespan_exclude "['lm_head','embed_tokens','visual']" \
  --lora_rank {LORA_RANK} --lora_alpha {LORA_ALPHA} --lora_dropout 0.05 \
  --num_lora_modules -1 \
  --freeze_llm True --freeze_vision_tower True --freeze_merger True \
  --fp16 True --bf16 False --tf32 False \
  --disable_flash_attn2 True --use_liger_kernel False \
  --output_dir "{OUTPUT_DIR}" \
  --num_train_epochs {EPOCHS} \
  --per_device_train_batch_size {MICRO_BATCH} \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --gradient_checkpointing True \
  --max_seq_length {MAX_SEQUENCE_LENGTH} \
  --image_min_pixels {MIN_IMAGE_PIXELS} \
  --image_max_pixels {MAX_IMAGE_PIXELS} \
  --learning_rate {LEARNING_RATE} \
  --optim paged_adamw_8bit \
  --weight_decay 0.01 --warmup_ratio 0.03 --lr_scheduler_type cosine \
  --eval_strategy epoch --prediction_loss_only True \
  --save_strategy epoch --save_total_limit 3 \
  --logging_steps 10 --logging_dir "{OUTPUT_DIR / 'logs'}" \
  --report_to tensorboard \
  --dataloader_num_workers 2 \
  --seed {SEED} --data_seed {SEED} \
  --load_best_model_at_end False
'''
launch_path = WORK / "launch_training.sh"
launch_path.write_text(launch)
launch_path.chmod(0o755)
print(launch)


In [ ]:
# Fixed-epoch training. Re-run this cell to resume after an interrupted session.
result = subprocess.run([str(launch_path)], text=True)
assert result.returncode == 0, f"Training failed with exit code {result.returncode}"
assert (OUTPUT_DIR / "adapter_config.json").exists(), "Final LoRA adapter was not saved."
print("Training complete:", OUTPUT_DIR)


In [ ]:
# Confirm the run really reached the requested epoch count and retain reproducibility metadata.
state = json.loads((OUTPUT_DIR / "trainer_state.json").read_text())
assert state["epoch"] >= EPOCHS, state["epoch"]
run_manifest = {
    "base_model": MODEL_ID,
    "dataset_archive": ARCHIVE.name,
    "dataset_sha256": archive_sha256,
    "epochs_requested": EPOCHS,
    "epochs_completed": state["epoch"],
    "global_steps": state["global_step"],
    "global_batch_size": 2 * MICRO_BATCH * GRAD_ACCUM,
    "early_stopping": False,
    "precision": "FP16 compute / NF4 base weights",
    "trainer_commit": expected_commit,
}
(OUTPUT_DIR / "thrivelens_run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print(json.dumps(run_manifest, indent=2))


## Held-out test

This launches both T4s again, loads the final adapter, partitions an untouched test subset between GPUs, generates predictions, and writes exact/contains metrics plus sample outputs. The default evaluates 5,000 deterministically shuffled held-out records to protect the weekly GPU budget. Set `MAX_TEST_SAMPLES = 0` to evaluate the entire large test split.


In [ ]:
# Deterministic held-out evaluation on both GPUs.
EVAL_SCRIPT = WORK / "training" / "evaluate_foodr1.py"
metrics_path = OUTPUT_DIR / "held_out_test_metrics.json"
cmd = [
    "torchrun", "--standalone", "--nproc_per_node=2", str(EVAL_SCRIPT),
    "--archive", str(ARCHIVE),
    "--test-json", str(WORK / "sft/test.json"),
    "--adapter", str(OUTPUT_DIR),
    "--model", MODEL_ID,
    "--max-samples", str(MAX_TEST_SAMPLES),
    "--output", str(metrics_path),
]
result = subprocess.run(cmd, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0,1"})
assert result.returncode == 0
metrics = json.loads(metrics_path.read_text())
print(json.dumps({k: v for k, v in metrics.items() if k != "sample_predictions"}, indent=2))


In [ ]:
# Package the adapter, processor, logs, run manifest, and test report for download.
deliverable = shutil.make_archive(
    "/kaggle/working/ThriveLens_FoodR1_QLoRA_Adapter",
    "zip",
    root_dir=OUTPUT_DIR,
)
print("Download from Kaggle Output:", deliverable, f"({Path(deliverable).stat().st_size / 2**20:.1f} MiB)")
